# VLM Salient Event Detection

**Computer Vision — Assignment 2, Group 6**

This notebook uses **SmolVLM2** to automatically detect salient events in a video.

It produces two types of output:
- **Event-only list** — what happened, in order
- **Event + timestamp list** — what happened and when

Both outputs are saved to files for use in the rest of the assignment.

---

## Section 1 — Setup

Install and import all required libraries.

In [84]:
# Install required packages
!pip install transformers accelerate --quiet
!pip install torch torchvision torchaudio --quiet
!pip install opencv-python pillow pandas num2words av --quiet
# sentence-transformers is used for semantic deduplication of events
!pip install sentence-transformers --quiet

Python(17338) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17339) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17340) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(17341) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [85]:
import torch
import cv2
import re
import json
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, AutoModelForImageTextToText
from sentence_transformers import SentenceTransformer

print("All libraries imported successfully!")

All libraries imported successfully!


## Section 2 — Configuration

Change `VIDEO_PATH` to point to your video file. All other settings have sensible defaults you can leave as-is.

In [86]:
# ── User settings ───────────────────────────────────────────────────────────
VIDEO_PATH = "video_21.mp4"   # <-- change this to your video file path

MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

# Frames SmolVLM2 samples internally in direct-video mode
NUM_FRAMES = 16

# Frames our seeker samples for the timestamp pipeline (uniform across full video)
N_FRAMES = 20

# Token budget — 600 gives room for up to ~18 detailed events
MAX_NEW_TOKENS = 600

# Semantic dedup: cosine similarity threshold (LOWERED from 0.80 to 0.70 for aggressive dedup)
SIM_THRESHOLD = 0.70

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def compute_target_events(duration_sec):
    """
    Return the target number of salient events based on video length.

    Scale:
        <= 2 min  →  5  (range 4–6)
        <= 5 min  →  8  (range 6–10)
        <= 10 min → 12  (range 10–15)
        > 10 min  → 15  (range 12–18)
    """
    minutes = duration_sec / 60
    if minutes <= 2:
        return 5
    elif minutes <= 5:
        return 8
    elif minutes <= 10:
        return 12
    else:
        return 15


print(f"Video  : {VIDEO_PATH}")
print(f"Model  : {MODEL_NAME}")
print(f"Frames : {NUM_FRAMES} (direct) | {N_FRAMES} (frame-based, full-video seeking)")
print(f"Tokens : {MAX_NEW_TOKENS}")
print(f"Dedup  : Similarity threshold = {SIM_THRESHOLD} (LOWERED for aggressive filtering)")

# Quick preview of the scaling table
print()
print("Dynamic event scaling:")
for d, label in [(60, "1 min"), (180, "3 min"), (420, "7 min"), (720, "12 min")]:
    print(f"  {label:>6} → {compute_target_events(d)} events")

Video  : video_21.mp4
Model  : HuggingFaceTB/SmolVLM2-2.2B-Instruct
Frames : 16 (direct) | 20 (frame-based, full-video seeking)
Tokens : 600
Dedup  : Similarity threshold = 0.7 (LOWERED for aggressive filtering)

Dynamic event scaling:
   1 min → 5 events
   3 min → 8 events
   7 min → 12 events
  12 min → 15 events


## Section 3 — Video Loading & Frame Sampling

We first inspect the video to know its duration and frame rate.

We also provide a helper that **uniformly samples one frame every N seconds** — this is used as a fallback if the model cannot ingest the raw video file directly.

In [87]:
def get_video_info(video_path):
    """Return (fps, total_frames, duration_seconds) for a video file."""
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur   = total / fps if fps > 0 else 0
    cap.release()
    return fps, total, dur


def sample_frames_uniform(video_path, n_frames=N_FRAMES):
    """
    Sample exactly `n_frames` frames spread UNIFORMLY across the ENTIRE video.

    Uses direct frame seeking (cap.set) instead of iterating every frame,
    so it is fast even for long videos and always covers the full duration.

    For a 10-min video with n_frames=20 the interval is ~32 seconds:
        Frame 1  →  0:00
        Frame 2  →  0:32
        ...
        Frame 20 →  10:08

    Returns:
        frames     : list of PIL.Image (RGB), length == n_frames
        timestamps : list of float seconds (the real time of each frame)
    """
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0 or fps == 0:
        cap.release()
        return [], []

    # Evenly-spaced frame indices across [0, total)
    indices = [int(total * i / n_frames) for i in range(n_frames)]

    frames, timestamps = [], []
    for frame_idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
            timestamps.append(round(frame_idx / fps, 2))

    cap.release()

    interval = (total / fps) / n_frames
    print(f"Sampled {len(frames)} frames uniformly across {total/fps:.0f}s "
          f"(~1 frame every {interval:.0f}s, from {timestamps[0]:.0f}s to {timestamps[-1]:.0f}s)")
    return frames, timestamps


fps, total_frames, duration = get_video_info(VIDEO_PATH)
print(f"Video    : {VIDEO_PATH}")
print(f"FPS      : {fps:.2f}  |  Total frames: {total_frames}  |  Duration: {duration:.1f}s ({duration/60:.1f} min)")
print()
# Preview what sampling will look like
print(f"With N_FRAMES={N_FRAMES}: 1 frame every ~{duration/N_FRAMES:.0f}s "
      f"covering the full {duration/60:.1f} minutes")

Video    : video_21.mp4
FPS      : 29.97  |  Total frames: 19406  |  Duration: 647.5s (10.8 min)

With N_FRAMES=20: 1 frame every ~32s covering the full 10.8 minutes


## Section 4 — Load the Vision-Language Model

We load **SmolVLM2-2.2B-Instruct** from HuggingFace.

- If a GPU is available, the model is loaded in **bfloat16** (half precision) to save memory.
- If only a CPU is available, it runs in **float32** (slower but works).

> The first download is ~4 GB. Subsequent runs use the local cache.

In [88]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"Loading '{MODEL_NAME}' on {DEVICE} ({DTYPE}) ...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)

print("Model loaded successfully!")

Loading 'HuggingFaceTB/SmolVLM2-2.2B-Instruct' on cpu (torch.float32) ...


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 17.23it/s]


Model loaded successfully!


## Section 5 — Prompt Design

We use two carefully structured prompts:

- **Prompt A** asks for a plain event list (no timestamps).
- **Prompt B** asks for the same list but with a `MM:SS - MM:SS` timestamp range for each event.

Both prompts enforce a strict numbered format so the parser can extract results reliably.

In [89]:
def build_event_only_prompt(duration_sec):
    """
    Build the event-only prompt dynamically from the actual video duration.
    Tells the model exactly how many events to generate and injects the duration.
    STRENGTHENED: Much more aggressive about preventing repetition.
    """
    minutes  = duration_sec / 60
    target   = compute_target_events(duration_sec)
    dur_str  = f"{minutes:.1f} minutes"

    return f"""Analyze the entire video (or sequence of frames) and extract ONLY the most important and distinct events.

CRITICAL RULES:
- Do NOT describe every frame — describe only major changes
- NEVER report the same action twice, even with slightly different wording
- If the person is "walking", "jogging", or "running" — pick ONE representative verb, do not repeat it
- If you describe "A man walks on a sidewalk" once, NEVER mention walking/running on sidewalk again
- Group all similar/repeated actions into a single event description
- Focus ONLY on meaningful transitions, not frame-by-frame descriptions

The video is {dur_str} long. Generate exactly {target} salient events, evenly distributed from the beginning to the end of the video.

Each event must:
- Be unique (fundamentally different from all others)
- Represent a clear change or transition
- Be written in one concise sentence
- Capture high-level actions, not low-level motion

Output format:
Salient event 1: ...
Salient event 2: ...
...

CONSTRAINTS:
- Do NOT repeat similar actions (walking, running, jogging are all the same action family)
- Do NOT produce variations of the same event
- Do NOT list frame-level details
- Generate EXACTLY {target} distinct events, no more, no less"""


def build_timestamp_prompt(duration_sec, ts_labels, frame_index_str):
    """
    Build the timestamp prompt dynamically.
    STRENGTHENED: Much more aggressive about preventing repetition.
    Injects the real sampled timestamps so the model can reference them.
    """
    minutes  = duration_sec / 60
    target   = compute_target_events(duration_sec)
    dur_str  = f"{minutes:.1f} minutes"

    return (
        f"You are analyzing {len(ts_labels)} frames sampled from a {dur_str} video.\n"
        f"Each frame was captured at this exact time:\n{frame_index_str}\n\n"
        f"CRITICAL RULES:\n"
        f"- Do NOT describe every frame — describe only major changes\n"
        f"- NEVER report the same action twice, even with slightly different wording\n"
        f"- If the person is 'walking', 'jogging', or 'running' — pick ONE verb, do not repeat it\n"
        f"- If you describe 'A man walks on a sidewalk' once, NEVER mention that action again\n"
        f"- Group all similar/repeated actions into a single event\n"
        f"- Focus ONLY on meaningful transitions\n\n"
        f"The video is {dur_str} long. Generate exactly {target} salient events, "
        f"evenly distributed from beginning to end.\n\n"
        f"Each event must:\n"
        f"- Be unique (fundamentally different from all others)\n"
        f"- Represent a clear change or transition\n"
        f"- Be written in one concise sentence\n"
        f"- Capture high-level actions, not low-level motion\n"
        f"- Include a timestamp range in MM:SS - MM:SS format\n\n"
        f"Output format:\n"
        f"Salient event 1: description, MM:SS - MM:SS\n"
        f"Salient event 2: description, MM:SS - MM:SS\n"
        f"...\n\n"
        f"IMPORTANT: Use ONLY timestamps from this list: {', '.join(ts_labels)}\n"
        f"Pick the closest frame time for event start and end.\n"
        f"Generate EXACTLY {target} distinct events with timestamps, no more, no less."
    )


print("Prompt builder functions defined (STRENGTHENED with aggressive anti-repetition rules).")
# Preview for current video
_, _, _dur = get_video_info(VIDEO_PATH)
_tgt = compute_target_events(_dur)
print(f"Current video : {_dur/60:.1f} min → {_tgt} target events")
print()
print("── Sample prompt (event-only) ─────────────────────────────────────")
print(build_event_only_prompt(_dur))

Prompt builder functions defined (STRENGTHENED with aggressive anti-repetition rules).
Current video : 10.8 min → 15 target events

── Sample prompt (event-only) ─────────────────────────────────────
Analyze the entire video (or sequence of frames) and extract ONLY the most important and distinct events.

CRITICAL RULES:
- Do NOT describe every frame — describe only major changes
- NEVER report the same action twice, even with slightly different wording
- If the person is "walking", "jogging", or "running" — pick ONE representative verb, do not repeat it
- If you describe "A man walks on a sidewalk" once, NEVER mention walking/running on sidewalk again
- Group all similar/repeated actions into a single event description
- Focus ONLY on meaningful transitions, not frame-by-frame descriptions

The video is 10.8 minutes long. Generate exactly 15 salient events, evenly distributed from the beginning to the end of the video.

Each event must:
- Be unique (fundamentally different from all ot

## Section 6 — VLM Inference Functions

Two different strategies are used, one per task:

| Task | Strategy | Why |
|------|----------|-----|
| **Event-only** | Direct video ingestion | Simpler; timestamps not needed |
| **Event + timestamps** | Frame-based with timestamps injected in prompt | SmolVLM2 has no temporal metadata in video mode — it cannot produce real `MM:SS` values unless you tell it when each frame was taken |

The raw model output will still be repetitive (the model describes each frame). Section 9 fixes this with **semantic deduplication**.

In [90]:
def _secs_to_mmss(seconds):
    """Convert float seconds → 'MM:SS' string."""
    m, s = divmod(int(seconds), 60)
    return f"{m:02d}:{s:02d}"


def _build_inputs_event_only(video_path, prompt, num_frames=NUM_FRAMES):
    """
    Build model inputs for event-only detection.
    Primary: direct video (SmolVLM2 samples num_frames uniformly internally).
    Fallback: manual uniform seeking across full video.
    """
    try:
        messages = [{
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text",  "text": prompt},
            ],
        }]
        inputs = processor.apply_chat_template(
            messages,
            num_frames=num_frames,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(DEVICE, dtype=DTYPE)
        print(f"  [video mode] {num_frames} frames sampled uniformly across full video")
        return inputs

    except Exception as e:
        print(f"  [video mode] Failed ({e}). Switching to frame mode ...")

    frames, _ = sample_frames_uniform(video_path, n_frames=num_frames)
    image_items = [{"type": "image", "url": f} for f in frames]
    messages = [{
        "role": "user",
        "content": image_items + [{"type": "text", "text": prompt}],
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE, dtype=DTYPE)
    print(f"  [frame mode] {len(frames)} frames (uniform, full-video)")
    return inputs


def _build_inputs_with_timestamps(video_path, n_frames=N_FRAMES):
    """
    Build model inputs for timestamp detection.
    Always frame-based: samples n_frames uniformly across the full video and
    injects real timestamps + dynamic target event count into the prompt.
    """
    frames, timestamps = sample_frames_uniform(video_path, n_frames=n_frames)
    ts_labels      = [_secs_to_mmss(t) for t in timestamps]
    frame_index_str = "\n".join([f"  Frame {i+1}: {ts}" for i, ts in enumerate(ts_labels)])

    _, _, duration = get_video_info(video_path)
    prompt = build_timestamp_prompt(duration, ts_labels, frame_index_str)

    image_items = [{"type": "image", "url": f} for f in frames]
    messages = [{
        "role": "user",
        "content": image_items + [{"type": "text", "text": prompt}],
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE, dtype=DTYPE)
    print(f"  [frame mode] {len(frames)} frames | {ts_labels[0]} → {ts_labels[-1]} (full video)")
    return inputs


def _decode(generated_ids):
    """Decode token IDs; strip echoed conversation, keep only the assistant reply."""
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    if "Assistant:" in text:
        text = text.split("Assistant:")[-1].strip()
    return text


print("Helper functions defined.")

Helper functions defined.


In [91]:
def run_vlm_event_only(video_path, num_frames=NUM_FRAMES):
    """
    Detect salient events without timestamps.

    Automatically computes video duration, selects the right event count,
    and builds a dynamic prompt before running inference.
    """
    _, _, duration = get_video_info(video_path)
    target  = compute_target_events(duration)
    prompt  = build_event_only_prompt(duration)

    print(f"Running event-only detection on '{video_path}' ...")
    print(f"  Duration : {duration:.1f}s ({duration/60:.1f} min)")
    print(f"  Target   : {target} events")

    inputs = _build_inputs_event_only(str(video_path), prompt, num_frames)
    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
    return _decode(generated_ids), target   # returns (raw_text, target_n)


print("run_vlm_event_only() defined.")

run_vlm_event_only() defined.


In [92]:
def run_vlm_event_with_timestamps(video_path, n_frames=N_FRAMES):
    """
    Detect salient events WITH timestamps.

    Uses frame-based mode: samples n_frames uniformly across the full video,
    injects real timestamps + dynamic event count into the prompt.
    """
    _, _, duration = get_video_info(video_path)
    target = compute_target_events(duration)

    print(f"Running event + timestamp detection on '{video_path}' ...")
    print(f"  Duration : {duration:.1f}s ({duration/60:.1f} min)")
    print(f"  Target   : {target} events")

    inputs = _build_inputs_with_timestamps(str(video_path), n_frames)
    generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS)
    return _decode(generated_ids), target   # returns (raw_text, target_n)


print("run_vlm_event_with_timestamps() defined.")

run_vlm_event_with_timestamps() defined.


## Section 7 — Run the Pipeline

> **Important:** If you have changed the prompts or inference functions, you **must re-run both cells below** to get fresh model output. The raw output will look repetitive — that is expected and is cleaned up in Section 9.

- Cell below runs ~3–5 min on CPU per call.
- On GPU: ~30–90 seconds per call.

In [93]:
event_only_output, target_events_only = run_vlm_event_only(VIDEO_PATH)

print()
print("=" * 65)
print("RAW OUTPUT — Event Only")
print("=" * 65)
print(event_only_output)

Running event-only detection on 'video_21.mp4' ...
  Duration : 647.5s (10.8 min)
  Target   : 15 events


/Users/spopa/Library/Python/3.9/lib/python/site-packages/transformers/video_processing_utils.py:879: UserWarning: `torchcodec` is not installed and cannot be used to decode the video by default. Falling back to `torchvision`. Note that `torchvision` decoding is deprecated and will be removed in future versions. 
  warnings.warn(
/Users/spopa/Library/Python/3.9/lib/python/site-packages/transformers/video_utils.py:524: UserWarning: Using `torchvision` for video decoding is deprecated and will be removed in future versions. Please use `torchcodec` instead.
  warnings.warn(
/Users/spopa/Library/Python/3.9/lib/python/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.war

  [video mode] 16 frames sampled uniformly across full video

RAW OUTPUT — Event Only
1. A man in a black shirt is walking on a sidewalk.
2. A group of people are playing a game of parkour on a rooftop.
3. A man is performing a parkour move on a rooftop.
4. A man is walking on a sidewalk in a park.
5. A man is walking on a sidewalk in a park.
6. A man is walking on a sidewalk in a park.
7. A man is walking on a sidewalk in a park.
8. A man is walking on a sidewalk in a park.
9. A man is walking on a sidewalk in a park.
10. A man is walking on a sidewalk in a park.
11. A man is walking on a sidewalk in a park.
12. A man is walking on a sidewalk in a park.
13. A man is walking on a sidewalk in a park.
14. A man is walking on a sidewalk in a park.
15. A man is walking on a sidewalk in a park.


In [94]:
event_timestamp_output, target_events_ts = run_vlm_event_with_timestamps(VIDEO_PATH)

print()
print("=" * 65)
print("RAW OUTPUT — Event + Timestamps")
print("=" * 65)
print(event_timestamp_output)

Running event + timestamp detection on 'video_21.mp4' ...
  Duration : 647.5s (10.8 min)
  Target   : 15 events
Sampled 20 frames uniformly across 648s (~1 frame every 32s, from 0s to 615s)


Token indices sequence length is longer than the specified maximum sequence length for this model (22346 > 16384). Running this sequence through the model will result in indexing errors


  [frame mode] 20 frames | 00:00 → 10:15 (full video)


KeyboardInterrupt: 

## Section 8 — Save Outputs

We save three files to the `outputs/` folder:

| File | Contents |
|------|----------|
| `vlm_event_only_output.txt` | Plain text — event list only |
| `vlm_event_timestamp_output.txt` | Plain text — events with timestamps |
| `vlm_outputs.json` | Both outputs together with metadata |

In [ ]:
# Save plain text outputs
(OUTPUT_DIR / "vlm_event_only_output.txt").write_text(event_only_output, encoding="utf-8")
(OUTPUT_DIR / "vlm_event_timestamp_output.txt").write_text(event_timestamp_output, encoding="utf-8")

# Save combined JSON with full metadata
result_dict = {
    "video_path":         VIDEO_PATH,
    "model_name":         MODEL_NAME,
    "event_only_prompt":  PROMPT_EVENT_ONLY,
    "event_only_output":  event_only_output,
    "timestamp_prompt":   PROMPT_EVENT_TIMESTAMP,
    "timestamp_output":   event_timestamp_output,
}

with open(OUTPUT_DIR / "vlm_outputs.json", "w", encoding="utf-8") as f:
    json.dump(result_dict, f, indent=2, ensure_ascii=False)

print("Files saved:")
for path in sorted(OUTPUT_DIR.iterdir()):
    size_kb = path.stat().st_size / 1024
    print(f"  {path}  ({size_kb:.1f} KB)")

## Section 9 — Parser

The raw model output is plain text. We parse it into structured **pandas DataFrames**.

- `event_only_df` — columns: `event_number`, `event_description`
- `timestamp_df`  — columns: `event_number`, `event_description`, `start_time`, `end_time`, `start_seconds`, `end_seconds`

The parser looks for lines matching the format the model was instructed to use, e.g.:
```
Salient event 1: A person enters the room.
Salient event 2: Two people shake hands, 00:12 - 00:18
```

In [ ]:
# ── Load embedding model once ───────────────────────────────────────────────
_embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Sentence embedder loaded.")

# Import DBSCAN for clustering-based deduplication
from sklearn.cluster import DBSCAN


# ── Semantic deduplication (clustering-based) ────────────────────────────

def _semantic_dedup(descriptions, max_events, sim_threshold=SIM_THRESHOLD):
    """
    Remove semantically near-duplicate events using DBSCAN clustering.
    
    Algorithm: 
    1. Encode all descriptions to embeddings
    2. Compute cosine distance matrix
    3. Use DBSCAN to group similar descriptions into clusters
    4. Pick ONE representative from each cluster (the most central one)
    5. Cap at max_events
    
    This is MORE AGGRESSIVE than greedy and handles cycles like:
        "A man walks" → "A man runs" → "A man walks" → "A man runs"
    by recognizing that "walks" and "runs" are very similar.
    """
    if not descriptions:
        return []

    # Encode and normalize
    embeddings = _embedder.encode(descriptions, convert_to_numpy=True, show_progress_bar=False)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.maximum(norms, 1e-8)
    
    # Compute cosine distance matrix (1 - cosine similarity)
    # DBSCAN uses distance, so we convert: distance = 1 - similarity
    dist_matrix = 1 - (embeddings @ embeddings.T)
    
    # DBSCAN clustering
    # eps = 1 - sim_threshold (e.g., if threshold is 0.70, eps is 0.30)
    eps = 1 - sim_threshold
    clustering = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    labels = clustering.fit_predict(dist_matrix)
    
    # Pick representative from each cluster
    unique_clusters = set(labels)
    cluster_reps = {}
    
    for cluster_id in sorted(unique_clusters):
        cluster_indices = np.where(labels == cluster_id)[0]
        
        # Pick the most central one (closest to cluster centroid)
        cluster_embeddings = embeddings[cluster_indices]
        centroid = cluster_embeddings.mean(axis=0)
        centroid_norm = np.linalg.norm(centroid)
        if centroid_norm > 0:
            centroid = centroid / centroid_norm
        
        # Find the description closest to centroid
        sims = cluster_embeddings @ centroid
        best_idx = cluster_indices[sims.argmax()]
        cluster_reps[cluster_id] = descriptions[best_idx]
    
    # Sort by cluster order and cap at max_events
    result = [cluster_reps[cid] for cid in sorted(unique_clusters)][:max_events]
    return result


# ── Extract raw lines from model output ────────────────────────────────────

def _extract_descriptions(text):
    """
    Extract event descriptions from raw model output.
    Handles both 'Salient event N: ...' and 'N. ...' formats.
    """
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    descriptions = []

    pattern_salient  = re.compile(r"[Ss]alient\s+event\s+\d+\s*[:\-]\s*(.+)", re.IGNORECASE)
    pattern_numbered = re.compile(r"^\d+\.\s+(.+)$")

    for line in lines:
        m = pattern_salient.search(line)
        if m:
            descriptions.append(m.group(1).strip())

    if not descriptions:
        for line in lines:
            m = pattern_numbered.match(line)
            if m:
                descriptions.append(m.group(1).strip())

    return descriptions


def _extract_ts_rows(text):
    """
    Extract (description, start, end) tuples from raw timestamp output.
    Handles both 'Salient event N: desc, MM:SS - MM:SS' and 'N. desc, MM:SS - MM:SS'.
    """
    lines  = [l.strip() for l in text.splitlines() if l.strip()]
    ts_pat = r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})"
    rows   = []

    pattern_salient  = re.compile(
        r"[Ss]alient\s+event\s+\d+\s*[:\-]\s*(.+?),\s*" + ts_pat, re.IGNORECASE
    )
    pattern_numbered = re.compile(r"^\d+\.\s+(.+?),\s*" + ts_pat)

    for line in lines:
        m = pattern_salient.search(line)
        if m:
            rows.append((m.group(1).strip(), m.group(2).strip(), m.group(3).strip()))

    if not rows:
        for line in lines:
            m = pattern_numbered.match(line)
            if m:
                rows.append((m.group(1).strip(), m.group(2).strip(), m.group(3).strip()))

    return rows


def _ts_to_seconds(ts_str):
    """'MM:SS' or 'HH:MM:SS' → total seconds (int). None on failure."""
    try:
        parts = [int(p) for p in ts_str.strip().split(":")]
        if len(parts) == 2:
            return parts[0] * 60 + parts[1]
        if len(parts) == 3:
            return parts[0] * 3600 + parts[1] * 60 + parts[2]
    except Exception:
        pass
    return None


# ── Public parse functions ──────────────────────────────────────────────────

def parse_event_only(text, max_events):
    """
    Parse raw event-only output → cleaned, deduplicated DataFrame.
    `max_events` is computed from video duration and controls the dedup cap.
    Uses CLUSTERING-BASED dedup (more aggressive than greedy).
    Columns: event_number, event_description
    """
    raw   = _extract_descriptions(text)
    clean = _semantic_dedup(raw, max_events=max_events)
    rows  = [{"event_number": i + 1, "event_description": d} for i, d in enumerate(clean)]
    return pd.DataFrame(rows, columns=["event_number", "event_description"])


def parse_event_timestamps(text, max_events):
    """
    Parse raw timestamp output → cleaned, deduplicated DataFrame.
    `max_events` is computed from video duration and controls the dedup cap.
    Uses CLUSTERING-BASED dedup (more aggressive than greedy).
    Columns: event_number, event_description, start_time, end_time, start_seconds, end_seconds
    """
    raw_rows = _extract_ts_rows(text)
    if not raw_rows:
        return pd.DataFrame(columns=["event_number", "event_description",
                                     "start_time", "end_time", "start_seconds", "end_seconds"])

    descs        = [r[0] for r in raw_rows]
    clean_descs  = _semantic_dedup(descs, max_events=max_events)

    desc_to_ts = {}
    for desc, start, end in raw_rows:
        if desc not in desc_to_ts:
            desc_to_ts[desc] = (start, end)

    rows = []
    for i, desc in enumerate(clean_descs):
        start, end = desc_to_ts.get(desc, ("N/A", "N/A"))
        rows.append({
            "event_number":      i + 1,
            "event_description": desc,
            "start_time":        start,
            "end_time":          end,
            "start_seconds":     _ts_to_seconds(start),
            "end_seconds":       _ts_to_seconds(end),
        })

    return pd.DataFrame(rows, columns=["event_number", "event_description",
                                       "start_time", "end_time",
                                       "start_seconds", "end_seconds"])


print("Parser functions defined (CLUSTERING-BASED dedup + strengthened prompts).")

In [ ]:
# ── Debug: raw model output ─────────────────────────────────────────────────
raw_only  = _extract_descriptions(event_only_output)
raw_ts    = _extract_ts_rows(event_timestamp_output)

print("=" * 65)
print(f"DEBUG — Raw descriptions extracted (event-only): {len(raw_only)} lines")
print("=" * 65)
for i, d in enumerate(raw_only, 1):
    print(f"  {i:>2}. {d}")

print()
print("=" * 65)
print(f"DEBUG — Raw descriptions extracted (timestamp): {len(raw_ts)} lines")
print("=" * 65)
for i, (d, s, e) in enumerate(raw_ts, 1):
    print(f"  {i:>2}. [{s} – {e}] {d}")

# ── Build cleaned DataFrames ────────────────────────────────────────────────
print()
print("Running semantic deduplication ...")
event_only_df = parse_event_only(event_only_output, target_events_only)
timestamp_df  = parse_event_timestamps(event_timestamp_output, target_events_ts)

print()
print("=" * 65)
print(f"CLEAN OUTPUT — Event Only: {len(raw_only)} raw → {len(event_only_df)} unique (target: {target_events_only})")
print("=" * 65)
for _, row in event_only_df.iterrows():
    print(f"  {row['event_number']:>2}. {row['event_description']}")

print()
print("=" * 65)
print(f"CLEAN OUTPUT — Timestamps: {len(raw_ts)} raw → {len(timestamp_df)} unique (target: {target_events_ts})")
print("=" * 65)
for _, row in timestamp_df.iterrows():
    print(f"  {row['event_number']:>2}. [{row['start_time']} – {row['end_time']}] {row['event_description']}")

if len(event_only_df) == 0:
    print()
    print("WARNING: No events parsed. Check the raw output in the cell above.")
    print("The model may have used an unexpected format.")

## Section 10 — Display Results

Show the raw model output and the parsed tables side by side.

In [ ]:
print("=" * 70)
print("RAW MODEL OUTPUT — Event Only")
print("=" * 70)
print(event_only_output)
print()
print("=" * 70)
print(f"CLEANED TABLE — {len(event_only_df)} Salient Events (after semantic dedup)")
print("=" * 70)
display(event_only_df)

In [ ]:
print("=" * 70)
print("RAW MODEL OUTPUT — Event + Timestamps")
print("=" * 70)
print(event_timestamp_output)
print()
print("=" * 70)
print(f"CLEANED TABLE — {len(timestamp_df)} Salient Events with Timestamps (after semantic dedup)")
print("=" * 70)
display(timestamp_df)

---

## Summary

After running this notebook end-to-end you will have:

| Output | Description |
|--------|-------------|
| `outputs/vlm_event_only_output.txt` | Raw text — event list |
| `outputs/vlm_event_timestamp_output.txt` | Raw text — events with timestamps |
| `outputs/vlm_outputs.json` | JSON with model metadata + both outputs |
| `event_only_df` | Pandas DataFrame — event number + description |
| `timestamp_df` | Pandas DataFrame — event number, description, start/end time |

These outputs form the basis for the assignment section:
> *"Retrieve all of the salient events of the video"*

**Next steps (not covered here):**
- Compare detected events against manual annotations.
- Evaluate with precision / recall / F1 on event coverage.
- Try different `NUM_FRAMES` values or temporal segmentation strategies.